# ETL Notebook (Simple)

Reads a CSV from a raw path, filters out rows where `value` is null, and writes a Delta output under a processed path organized by `run_date`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from datetime import date

spark = SparkSession.builder.getOrCreate()


In [ ]:
# Parameter resolution (job widgets > control table > defaults)

DEFAULT_ENV = 'dev'
DEFAULT_RAW_BASE_PATH = '/Volumes/workspace/default/raw'
DEFAULT_PROCESSED_BASE_PATH = '/Volumes/workspace/default/processed'
DEFAULT_INPUT_FILENAME = 'input.csv'

# Local parameter resolver (no external imports)
def get_param(spark, env, key, explicit=None, default=None):
    if explicit is not None and str(explicit).strip() != '':
        return explicit
    try:
        df = spark.read.table('workspace.default.control_parameters')
        row = df.filter((df.env == env) & (df.key == key)).select('value').first()
        if row:
            return row.value
    except Exception:
        pass
    return default

# Define widgets so job base_parameters populate them
dbutils.widgets.text('run_date', '')
dbutils.widgets.text('env', '')
dbutils.widgets.text('input_filename', '')

wd_run_date = dbutils.widgets.get('run_date')
wd_env = dbutils.widgets.get('env')
wd_input_filename = dbutils.widgets.get('input_filename')

# Resolve parameters
env = (wd_env.strip() if wd_env and wd_env.strip() != '' else DEFAULT_ENV)
run_date = get_param(spark, env, 'run_date', explicit=(wd_run_date.strip() if wd_run_date and wd_run_date.strip() != '' else None), default=date.today().strftime('%Y-%m-%d'))
raw_base_path = DEFAULT_RAW_BASE_PATH
processed_base_path = DEFAULT_PROCESSED_BASE_PATH
input_filename = get_param(spark, env, 'input_filename', explicit=(wd_input_filename.strip() if wd_input_filename and wd_input_filename.strip() != '' else None), default=DEFAULT_INPUT_FILENAME)

print(f'env={env}, run_date={run_date}')
print(f'raw_base_path={raw_base_path}, processed_base_path={processed_base_path}, input={input_filename}')


In [ ]:
# ETL
input_path = f"{raw_base_path}/{input_filename}"
parent_date_dir = f"{processed_base_path}/{run_date}"
output_path = f"{parent_date_dir}/Notebook"

df = spark.read.option('header', True).csv(input_path)
df_filtered = df.filter(col('value').isNotNull())

print(f'Read {df.count()} rows; writing {df_filtered.count()} non-null rows to {output_path}')

df_filtered.write.format('delta').mode('overwrite').save(output_path)

# Sanity: list the parent directory to confirm presence in UI
try:
    children = [f.name for f in dbutils.fs.ls(parent_date_dir)]  # type: ignore
    print(f'Parent contents after write: {children}')
except Exception as e:
    print(f'Listing parent failed: {e}')
